# Mininet PCAP Training & Prediction Pipeline (Google Colab)

**Complete Pipeline:**
1. Upload PCAP files from Mininet
2. Extract network flow features
3. Train ML models (RF, XGBoost, Ensemble)
4. Evaluate with comprehensive metrics
5. Save trained models
6. **Test predictions on NEW PCAP files**

**For Google Colab** - Includes file upload and download functionality

## Step 1: Install Dependencies

In [ ]:
!pip install -q scapy pandas numpy scikit-learn xgboost imbalanced-learn matplotlib seaborn
print("✓ All dependencies installed")

## Step 2: Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import defaultdict
import joblib

# Colab file handling
from google.colab import files

# Scapy for PCAP parsing
from scapy.all import rdpcap, IP, TCP, UDP, ICMP

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE

print("✓ All libraries imported")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 3: Upload Training PCAP Files

Upload your Mininet PCAP files:
- Normal traffic PCAP
- Attack PCAP files (SYN flood, port scan, UDP flood, HTTP flood)

In [ ]:
print("Upload your PCAP files...")
print("Expected files:")
print("  1. normal_traffic.pcap (or similar)")
print("  2. syn_flood.pcap")
print("  3. port_scan.pcap")
print("  4. udp_flood.pcap")
print("  5. http_flood.pcap")
print("\nClick 'Choose Files' and select all PCAP files...\n")

uploaded = files.upload()

print(f"\n✓ Uploaded {len(uploaded)} files:")
for filename in uploaded.keys():
    print(f"  - {filename} ({len(uploaded[filename])} bytes)")

## Step 4: Feature Extraction Class

In [ ]:
class PCAPFeatureExtractor:
    """Extract ML features from PCAP files"""
    
    def extract_from_pcap(self, pcap_file, label, attack_type):
        """Extract features from a single PCAP file"""
        print(f"\nProcessing: {pcap_file}")
        print(f"  Label: {label} ({attack_type})")
        
        try:
            packets = rdpcap(pcap_file)
            print(f"  Packets: {len(packets)}")
        except Exception as e:
            print(f"  ❌ Error: {e}")
            return []
        
        # Group packets by flow
        flows = defaultdict(list)
        for pkt in packets:
            if IP in pkt:
                flow_key = self._get_flow_key(pkt)
                if flow_key:
                    flows[flow_key].append(pkt)
        
        print(f"  Flows: {len(flows)}")
        
        # Extract features
        features = []
        for flow_key, flow_packets in flows.items():
            feature = self._extract_flow_features(flow_key, flow_packets, label, attack_type)
            if feature:
                features.append(feature)
        
        print(f"  ✓ Extracted {len(features)} flows")
        return features
    
    def _get_flow_key(self, pkt):
        if IP not in pkt:
            return None
        
        src_ip, dst_ip = pkt[IP].src, pkt[IP].dst
        
        if TCP in pkt:
            return (src_ip, dst_ip, pkt[TCP].sport, pkt[TCP].dport, 'TCP')
        elif UDP in pkt:
            return (src_ip, dst_ip, pkt[UDP].sport, pkt[UDP].dport, 'UDP')
        elif ICMP in pkt:
            return (src_ip, dst_ip, 0, 0, 'ICMP')
        return None
    
    def _extract_flow_features(self, flow_key, packets, label, attack_type):
        src_ip, dst_ip, src_port, dst_port, protocol = flow_key
        
        if len(packets) == 0:
            return None
        
        # Timing
        timestamps = [float(pkt.time) for pkt in packets]
        duration = max(timestamps) - min(timestamps) if len(timestamps) > 1 else 0.001
        
        # Sizes
        packet_sizes = [len(pkt) for pkt in packets]
        packet_count = len(packets)
        byte_count = sum(packet_sizes)
        
        # TCP flags
        syn_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x02)
        fin_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x01)
        rst_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x04)
        psh_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x08)
        ack_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x10)
        
        # Derived features
        packets_per_sec = packet_count / duration
        bytes_per_sec = byte_count / duration
        mean_packet_size = np.mean(packet_sizes)
        std_packet_size = np.std(packet_sizes) if len(packet_sizes) > 1 else 0
        
        # Inter-arrival times
        if len(timestamps) > 1:
            iat = np.diff(timestamps)
            mean_iat, std_iat = np.mean(iat), np.std(iat)
        else:
            mean_iat, std_iat = 0, 0
        
        return {
            'duration': duration,
            'protocol': protocol,
            'src_port': src_port,
            'dst_port': dst_port,
            'packet_count': packet_count,
            'byte_count': byte_count,
            'packets_per_sec': packets_per_sec,
            'bytes_per_sec': bytes_per_sec,
            'mean_packet_size': mean_packet_size,
            'std_packet_size': std_packet_size,
            'min_packet_size': min(packet_sizes),
            'max_packet_size': max(packet_sizes),
            'mean_inter_arrival_time': mean_iat,
            'std_inter_arrival_time': std_iat,
            'syn_count': syn_count,
            'fin_count': fin_count,
            'rst_count': rst_count,
            'psh_count': psh_count,
            'ack_count': ack_count,
            'syn_ratio': syn_count / packet_count,
            'fin_ratio': fin_count / packet_count,
            'rst_ratio': rst_count / packet_count,
            'psh_ratio': psh_count / packet_count,
            'ack_ratio': ack_count / packet_count,
            'is_well_known_port': 1 if dst_port < 1024 else 0,
            'label': label,
            'attack_type': attack_type
        }

print("✓ Feature extractor defined")

## Step 5: Process Uploaded PCAP Files

In [ ]:
print("="*60)
print("PROCESSING PCAP FILES")
print("="*60)

extractor = PCAPFeatureExtractor()
all_features = []

# Map uploaded files to labels
# Adjust these mappings based on your file names
file_mappings = {
    'normal': (0, 'normal'),
    'syn_flood': (1, 'syn_flood'),
    'port_scan': (1, 'port_scan'),
    'udp_flood': (1, 'udp_flood'),
    'http_flood': (1, 'http_flood')
}

# Process each uploaded file
for filename in uploaded.keys():
    # Determine label based on filename
    label, attack_type = None, None
    for key, (lbl, att) in file_mappings.items():
        if key in filename.lower():
            label, attack_type = lbl, att
            break
    
    if label is not None:
        features = extractor.extract_from_pcap(filename, label, attack_type)
        all_features.extend(features)
    else:
        print(f"\n⚠ Skipping {filename} - couldn't determine type")

# Convert to DataFrame
df = pd.DataFrame(all_features)

print(f"\n{'='*60}")
print(f"✓ Total samples: {len(df):,}")
print(f"  Normal: {len(df[df['label'] == 0]):,}")
print(f"  Attack: {len(df[df['label'] == 1]):,}")
print(f"  Features: {len(df.columns)}")
print("="*60)

df.head()

## Step 6: Data Preprocessing

In [ ]:
print("\nPreprocessing data...")

# Separate features and labels
X = df.drop(['label', 'attack_type'], axis=1)
y = df['label']

# Encode categorical features
label_encoders = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

# Handle missing/infinite values
X = X.fillna(0).replace([np.inf, -np.inf], 0)

print(f"✓ Features: {len(X.columns)}")
print(f"✓ Samples: {len(X)}")

## Step 7: Train/Val/Test Split & Feature Engineering

In [ ]:
# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Feature selection
k_features = min(30, X_train.shape[1])
selector = SelectKBest(mutual_info_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()
print(f"✓ Selected {len(selected_features)} features")

# SMOTE balancing
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)
print(f"✓ Balanced training set: {len(X_train_balanced)} samples")

## Step 8: Train Models

In [ ]:
print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf_model.fit(X_train_balanced, y_train_balanced)
print("✓ Random Forest trained")

print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100, max_depth=8, learning_rate=0.05,
    subsample=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1
)
xgb_model.fit(X_train_balanced, y_train_balanced)
print("✓ XGBoost trained")

print("\nCreating Ensemble...")
ensemble_model = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='soft', n_jobs=-1
)
ensemble_model.fit(X_train_balanced, y_train_balanced)
print("✓ Ensemble created")

## Step 9: Comprehensive Evaluation

In [ ]:
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# Predictions
models = {
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'Ensemble': ensemble_model
}

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test_selected)
    y_pred_proba = model.predict_proba(X_test_selected)[:, 1]
    
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }

results_df = pd.DataFrame(results).T
print("\nPerformance Comparison:")
print(results_df)

# Detailed metrics for ensemble
y_pred_ensemble = ensemble_model.predict(X_test_selected)
print("\n" + "="*60)
print("ENSEMBLE MODEL - DETAILED METRICS")
print("="*60)
print(classification_report(y_test, y_pred_ensemble, target_names=['Normal', 'Attack']))

cm = confusion_matrix(y_test, y_pred_ensemble)
print(f"\nConfusion Matrix:")
print(f"  TN: {cm[0,0]}, FP: {cm[0,1]}")
print(f"  FN: {cm[1,0]}, TP: {cm[1,1]}")

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title('Ensemble Model - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

## Step 10: Save Trained Models

In [ ]:
print("\nSaving models...")

# Save all models and components
joblib.dump(ensemble_model, 'mininet_ensemble_model.pkl')
joblib.dump(rf_model, 'mininet_random_forest_model.pkl')
joblib.dump(xgb_model, 'mininet_xgboost_model.pkl')
joblib.dump(scaler, 'mininet_scaler.pkl')
joblib.dump(selector, 'mininet_feature_selector.pkl')
joblib.dump(selected_features, 'mininet_feature_columns.pkl')
joblib.dump(label_encoders, 'mininet_label_encoders.pkl')

# Save metadata
metadata = {
    'training_date': datetime.now().isoformat(),
    'n_samples': len(df),
    'n_features': len(selected_features),
    'selected_features': selected_features,
    'metrics': results['Ensemble']
}
joblib.dump(metadata, 'mininet_model_metadata.pkl')

print("✓ Saved 8 model files")
print("\nFiles ready for download:")
print("  1. mininet_ensemble_model.pkl")
print("  2. mininet_random_forest_model.pkl")
print("  3. mininet_xgboost_model.pkl")
print("  4. mininet_scaler.pkl")
print("  5. mininet_feature_selector.pkl")
print("  6. mininet_feature_columns.pkl")
print("  7. mininet_label_encoders.pkl")
print("  8. mininet_model_metadata.pkl")

## Step 11: Download Trained Models

In [ ]:
print("Downloading trained models...")

model_files = [
    'mininet_ensemble_model.pkl',
    'mininet_random_forest_model.pkl',
    'mininet_xgboost_model.pkl',
    'mininet_scaler.pkl',
    'mininet_feature_selector.pkl',
    'mininet_feature_columns.pkl',
    'mininet_label_encoders.pkl',
    'mininet_model_metadata.pkl'
]

for file in model_files:
    if os.path.exists(file):
        files.download(file)
        print(f"  ✓ Downloaded: {file}")

print("\n✓ All models downloaded!")

## Step 12: TEST PREDICTIONS ON NEW PCAP FILES

**Upload NEW PCAP files** to test the trained models

In [ ]:
print("="*60)
print("PREDICTION ON NEW PCAP FILES")
print("="*60)
print("\nUpload NEW PCAP files to test predictions...\n")

new_pcaps = files.upload()

print(f"\n✓ Uploaded {len(new_pcaps)} new PCAP files")

## Step 13: Extract Features from New PCAP Files

In [ ]:
print("\nExtracting features from new PCAP files...")

new_features = []
new_extractor = PCAPFeatureExtractor()

for filename in new_pcaps.keys():
    # Extract features (label doesn't matter for prediction)
    features = new_extractor.extract_from_pcap(filename, label=0, attack_type='unknown')
    new_features.extend(features)

# Convert to DataFrame
new_df = pd.DataFrame(new_features)
print(f"\n✓ Extracted {len(new_df)} flows from new PCAP files")

# Prepare for prediction
X_new = new_df.drop(['label', 'attack_type'], axis=1, errors='ignore')

# Encode categorical features using saved encoders
for col in X_new.select_dtypes(include=['object']).columns:
    if col in label_encoders:
        le = label_encoders[col]
        X_new[col] = X_new[col].apply(lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else -1)
    else:
        X_new[col] = LabelEncoder().fit_transform(X_new[col].astype(str))

# Handle missing/infinite values
X_new = X_new.fillna(0).replace([np.inf, -np.inf], 0)

# Ensure same columns as training
for col in X.columns:
    if col not in X_new.columns:
        X_new[col] = 0
X_new = X_new[X.columns]

print("✓ Features prepared for prediction")

## Step 14: Make Predictions

In [ ]:
print("\n" + "="*60)
print("MAKING PREDICTIONS")
print("="*60)

# Scale and select features
X_new_scaled = scaler.transform(X_new)
X_new_selected = selector.transform(X_new_scaled)

# Predict with ensemble model
predictions = ensemble_model.predict(X_new_selected)
prediction_probas = ensemble_model.predict_proba(X_new_selected)

# Add predictions to DataFrame
new_df['prediction'] = predictions
new_df['prediction_label'] = new_df['prediction'].map({0: 'Normal', 1: 'Attack'})
new_df['confidence'] = prediction_probas.max(axis=1)

print(f"\n✓ Predictions complete!")
print(f"\nPrediction Summary:")
print(f"  Total flows: {len(predictions)}")
print(f"  Predicted Normal: {sum(predictions == 0)}")
print(f"  Predicted Attack: {sum(predictions == 1)}")
print(f"  Attack Rate: {sum(predictions == 1) / len(predictions) * 100:.1f}%")

# Show sample predictions
print("\nSample Predictions:")
display_cols = ['protocol', 'src_port', 'dst_port', 'packet_count', 'packets_per_sec', 
                'prediction_label', 'confidence']
print(new_df[display_cols].head(10))

## Step 15: Detailed Prediction Analysis

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Prediction distribution
ax = axes[0]
new_df['prediction_label'].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Prediction Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediction')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Confidence distribution
ax = axes[1]
new_df['confidence'].hist(bins=20, ax=ax, color='skyblue', edgecolor='black')
ax.set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Confidence')
ax.set_ylabel('Frequency')
ax.axvline(0.9, color='red', linestyle='--', label='High Confidence (>0.9)')
ax.legend()

plt.tight_layout()
plt.savefig('prediction_analysis.png', dpi=300)
plt.show()

# High-confidence attacks
high_conf_attacks = new_df[(new_df['prediction'] == 1) & (new_df['confidence'] > 0.9)]
print(f"\nHigh-Confidence Attacks (>90%): {len(high_conf_attacks)}")

if len(high_conf_attacks) > 0:
    print("\nTop 5 High-Confidence Attacks:")
    display_cols = ['protocol', 'src_port', 'dst_port', 'packet_count', 'packets_per_sec', 
                    'syn_ratio', 'confidence']
    print(high_conf_attacks[display_cols].head())

## Step 16: Export Predictions

In [ ]:
# Save predictions to CSV
output_file = f'predictions_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
new_df.to_csv(output_file, index=False)

print(f"✓ Predictions saved to: {output_file}")
print("\nDownloading predictions...")
files.download(output_file)

print("\n" + "="*60)
print("✓ PREDICTION PIPELINE COMPLETE!")
print("="*60)
print("\nSummary:")
print(f"  Trained on: {len(df)} samples")
print(f"  Model Accuracy: {results['Ensemble']['accuracy']:.4f}")
print(f"  Predicted on: {len(new_df)} new flows")
print(f"  Detected Attacks: {sum(predictions == 1)}")
print("\n✓ Models and predictions downloaded!")
print("="*60)